# Обзвон клиентов: выгрузка май–август 2026

Задача: отчёт по клиентам с коммуникацией (июнь/июль/август).

**Источники**
- Флаги коммуникации: `inn_list_june/july/august.txt`
- ЧОД ТЭ: `final_df` (май–август), `SUM(chod)` по ИНН×месяц
- Общий ЧОД: `kedr_obshiy_chod` (май–июль); **август не нужен**

**Перед выгрузкой**
1. Amort: `01_07_build_amortization_drp.ipynb` → таблица `…_amortization_model_jan_aug`
2. `final_df` август: `01_07_acq_dash_jan_jun_mpos.ipynb` с `period_end = '2026-08-01'`, `force_recompute_final_df = False`, `run_kedr_obshiy_chod_enrich = True`
3. Kedr lake за май–июль уже должен быть; **не** добавлять `202608`
4. В этой тетрадке секция **0b** проверяет CSV (есть `2026-08` и Общий ЧОД июля) — при ошибке выгрузка остановится


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s


def month_key(value):
    s = str(value).strip()
    if not s or s.lower() in {'nan', 'none', 'nat'}:
        return None
    ts = pd.to_datetime(s, errors='coerce')
    if pd.isna(ts):
        m = re.match(r'^(\d{4}-\d{2})', s)
        return m.group(1) if m else None
    return ts.strftime('%Y-%m')


# Стандартное подключение к Impala (как в final_script_2.ipynb)
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')


## 0) Пути и проверка артефактов

In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
ADHOC_DIR = Path('/home/jovyan/documents/Equaring/Adhocs/Скоринг клиентов обзвон')
OUT_DIR = ADHOC_DIR / 'exports'
OUT_DIR.mkdir(parents=True, exist_ok=True)

INN_LIST_PATHS = {
    '2026-06': ADHOC_DIR / 'inn_list_june.txt',
    '2026-07': ADHOC_DIR / 'inn_list_july.txt',
    '2026-08': ADHOC_DIR / 'inn_list_august.txt',
}

# Prefer Jan–Aug period CSV; fall back to Jan–Jul + August checkpoint
FINAL_DF_CANDIDATES = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
]

AUGUST_CHECKPOINT_CANDIDATES = [
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos' / 'final_df_2026_08.parquet',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos' / 'final_df_2026_08.csv',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_07_final_script_2' / 'final_df_2026_08.parquet',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_07_final_script_2' / 'final_df_2026_08.csv',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_mpos' / 'final_df_2026_08.parquet',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_mpos' / 'final_df_2026_08.csv',
]

KEDR_TABLE = 'sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month'
NEED_CHOD_TE_MONTHS = ['2026-05', '2026-06', '2026-07', '2026-08']
NEED_OBSHIY_MONTHS = ['2026-05', '2026-06', '2026-07']

print('=== inn_list ===')
for m, p in INN_LIST_PATHS.items():
    print(f'  {m}: exists={p.exists()} | {p}')

print('\n=== final_df candidates ===')
for p in FINAL_DF_CANDIDATES:
    print(f'  exists={p.exists()} | {p}')

print('\n=== August checkpoint candidates ===')
for p in AUGUST_CHECKPOINT_CANDIDATES:
    print(f'  exists={p.exists()} | {p}')


## 0b) Проверка CSV `final_df` (август + Общий ЧОД июля)

Перед выгрузкой убеждаемся, что period CSV готов:
- есть месяц `2026-08` (ЧОД ТЭ августа);
- колонка `kedr_obshiy_chod` заполнена за `2026-07`.

Если проверки падают — сначала догоните `01_07_acq_dash_jan_jun_mpos.ipynb` (с enrich), затем вернитесь сюда.

In [ ]:
# Проверка актуального final_df period CSV (до выгрузки обзвона)
csv_check_path = next((p for p in FINAL_DF_CANDIDATES if p.exists()), None)
if csv_check_path is None:
    raise FileNotFoundError(
        'Нет final_df period CSV. Ожидается final_df_period_2026_01_2026_08_mpos.csv '
        '(или …_07 + checkpoint августа). Прогони 01_07_acq_dash_jan_jun_mpos.ipynb.'
    )

csv_check_df = pd.read_csv(csv_check_path, low_memory=False)
csv_check_df.columns = [str(c).strip() for c in csv_check_df.columns]
csv_check_df['report_month'] = csv_check_df['report_month'].map(month_key)

print('CSV:', csv_check_path)
print('rows:', f'{len(csv_check_df):,}')
months_in_csv = sorted(csv_check_df['report_month'].dropna().unique().tolist())
print('months:', months_in_csv)

need_months = ['2026-05', '2026-06', '2026-07', '2026-08']
missing_months = [m for m in need_months if m not in set(months_in_csv)]
print('missing needed months:', missing_months)

has_kedr_col = 'kedr_obshiy_chod' in csv_check_df.columns
print('has kedr_obshiy_chod:', has_kedr_col)

july_kedr_inns = 0
july_rows = 0
if has_kedr_col:
    csv_check_df['kedr_obshiy_chod'] = pd.to_numeric(csv_check_df['kedr_obshiy_chod'], errors='coerce')
    jul = csv_check_df.loc[csv_check_df['report_month'] == '2026-07'].copy()
    july_rows = len(jul)
    july_kedr_inns = jul.loc[jul['kedr_obshiy_chod'].notna(), 'inn'].nunique() if 'inn' in jul.columns else 0
    print('July rows:', july_rows)
    print('July kedr non-null INNs:', july_kedr_inns)

# Monthly coverage snapshot
cov_rows = []
for m in months_in_csv:
    part = csv_check_df.loc[csv_check_df['report_month'] == m]
    chod_nn = pd.to_numeric(part.get('chod'), errors='coerce').notna().sum() if 'chod' in part.columns else 0
    kedr_nn = (
        pd.to_numeric(part.get('kedr_obshiy_chod'), errors='coerce').notna().sum()
        if has_kedr_col else 0
    )
    cov_rows.append({
        'report_month': m,
        'rows': len(part),
        'chod_nonnull': int(chod_nn),
        'kedr_nonnull': int(kedr_nn),
    })
csv_coverage_df = pd.DataFrame(cov_rows)
display(csv_coverage_df)

errors = []
if '2026-08' not in set(months_in_csv):
    errors.append('В CSV нет месяца 2026-08 (нужен ЧОД ТЭ августа)')
if missing_months:
    errors.append(f'Нет месяцев для отчёта: {missing_months}')
if not has_kedr_col:
    errors.append('Нет колонки kedr_obshiy_chod — не был enrich')
elif july_kedr_inns <= 0:
    errors.append('Общий ЧОД за июль пустой (July kedr non-null INNs = 0)')

if errors:
    raise RuntimeError(
        'Проверка CSV не пройдена:\n- ' + '\n- '.join(errors) + '\n'
        'Сначала 01_07_acq_dash_jan_jun_mpos.ipynb '
        '(period_end=2026-08-01, run_kedr_obshiy_chod_enrich=True).'
    )

print('OK: CSV check passed — август есть, Общий ЧОД июля заполнен')

In [ ]:
# Impala probe for Kedr months 202605–202607 (uses `imp` from setup cell)
check_kedr_in_impala = True
kedr_months_ok = None
kedr_probe_df = None

if check_kedr_in_impala:
    try:
        yearmms = [202605, 202606, 202607]
        sql = f'''
        select
          yearmm,
          count(*) as rows_cnt,
          count(distinct cast(inn as string)) as inns_cnt,
          sum(case when kedr_obshiy_chod is not null then 1 else 0 end) as filled_rows
        from {KEDR_TABLE}
        where yearmm in ({', '.join(str(y) for y in yearmms)})
        group by yearmm
        order by yearmm
        '''
        with imp:
            imp.execute('set MEM_LIMIT=4g')
            kedr_probe_df = imp.fetch(sql)
        print('=== Kedr lake probe ===')
        display(kedr_probe_df)
        present = set()
        if kedr_probe_df is not None and len(kedr_probe_df):
            present = set(int(x) for x in kedr_probe_df['yearmm'].tolist())
        kedr_months_ok = all(y in present for y in yearmms)
        print('kedr_months_ok (202605–202607):', kedr_months_ok)
        if kedr_probe_df is not None and len(kedr_probe_df):
            jul = kedr_probe_df.loc[kedr_probe_df['yearmm'].astype(int) == 202607]
            if jul.empty:
                print('WARN: yearmm=202607 отсутствует в lake — Общий ЧОД июля нет')
            else:
                print('July filled_rows:', int(jul['filled_rows'].iloc[0]))
    except Exception as exc:
        print(f'SKIP Kedr Impala probe: {type(exc).__name__}: {exc}')
        print('Можно продолжить, если в final_df уже есть колонка kedr_obshiy_chod.')
else:
    print('SKIP Kedr Impala probe (check_kedr_in_impala=False)')


## 1) Загрузка final_df (май–август)

Если period CSV уже содержит август — берём его.
Если только Jan–Jul — доклеиваем checkpoint августа (после прогона пайплайна с `period_end='2026-08-01'`).


In [ ]:
def load_final_df_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.parquet':
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, dtype=str, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def prepare_final_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'inn' not in out.columns:
        raise RuntimeError(f'final_df missing inn; cols={list(out.columns)[:30]}')
    out['inn'] = out['inn'].map(normalize_inn)
    out['report_month'] = out['report_month'].map(month_key)
    if 'chod' in out.columns:
        out['chod'] = pd.to_numeric(out['chod'], errors='coerce')
    else:
        out['chod'] = np.nan
    if 'kedr_obshiy_chod' in out.columns:
        out['kedr_obshiy_chod'] = pd.to_numeric(out['kedr_obshiy_chod'], errors='coerce')
    else:
        out['kedr_obshiy_chod'] = np.nan
    for c in ['filial_rf', 'company_name', 'tariff_short']:
        if c not in out.columns:
            out[c] = np.nan
    return out


final_df_path = next((p for p in FINAL_DF_CANDIDATES if p.exists()), None)
if final_df_path is None:
    raise FileNotFoundError(
        'Не найден final_df period CSV. Сначала прогони 01_07_acq_dash_jan_jun_mpos.ipynb '
        '(или final_script_2.ipynb).'
    )

final_df = prepare_final_df(load_final_df_table(final_df_path))
print(f'Loaded: {final_df_path}')
print('rows=', f'{len(final_df):,}', '| months=', sorted(final_df['report_month'].dropna().unique().tolist()))

months_present = set(final_df['report_month'].dropna().unique())
missing_te = [m for m in NEED_CHOD_TE_MONTHS if m not in months_present]
print('missing ЧОД ТЭ months:', missing_te)

if '2026-08' in missing_te:
    aug_path = next((p for p in AUGUST_CHECKPOINT_CANDIDATES if p.exists()), None)
    if aug_path is None:
        raise FileNotFoundError(
            'Нет августа в final_df и нет checkpoint final_df_2026_08.*\n'
            'Сделай:\n'
            '1) 01_07_build_amortization_drp.ipynb → period_end_exclusive="2026-09-01" → Run All\n'
            '2) 01_07_acq_dash_jan_jun_mpos.ipynb → period_end="2026-08-01", '
            'force_recompute_final_df=False, wipe_checkpoints_on_force=False → Restart & Run All\n'
            'Kedr на август НЕ нужен.'
        )
    aug_df = prepare_final_df(load_final_df_table(aug_path))
    if 'report_month' in aug_df.columns:
        aug_df = aug_df.loc[aug_df['report_month'].fillna('2026-08') == '2026-08'].copy()
    if aug_df.empty:
        raise RuntimeError(f'August checkpoint loaded but empty after filter: {aug_path}')
    if aug_df['report_month'].isna().all():
        aug_df['report_month'] = '2026-08'
    final_df = pd.concat([final_df, aug_df], ignore_index=True)
    print(f'Appended August from checkpoint: {aug_path} | rows={len(aug_df):,}')
    months_present = set(final_df['report_month'].dropna().unique())
    missing_te = [m for m in NEED_CHOD_TE_MONTHS if m not in months_present]
    print('missing ЧОД ТЭ months after append:', missing_te)

if missing_te:
    raise RuntimeError(f'Still missing months for ЧОД ТЭ: {missing_te}')

has_kedr_col = final_df['kedr_obshiy_chod'].notna().any()
print('kedr_obshiy_chod present in final_df:', has_kedr_col)
if not has_kedr_col:
    print('WARN: нет kedr_obshiy_chod в CSV — попробуем подтянуть из lake в следующей ячейке')


## 1b) Fallback: подтянуть Общий ЧОД из lake (май–июль), если нет в CSV

In [ ]:
need_kedr_join = (not has_kedr_col) or any(
    final_df.loc[final_df['report_month'] == m, 'kedr_obshiy_chod'].isna().all()
    for m in NEED_OBSHIY_MONTHS
    if m in set(final_df['report_month'])
)

if need_kedr_join:
    try:
        yearmms = [int(m.replace('-', '')) for m in NEED_OBSHIY_MONTHS]
        sql = f'''
        select
          cast(inn as string) as inn,
          cast(yearmm as bigint) as yearmm,
          cast(kedr_obshiy_chod as double) as kedr_obshiy_chod
        from {KEDR_TABLE}
        where yearmm in ({', '.join(map(str, yearmms))})
        '''
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            kedr_df = imp.fetch(sql)
        kedr_df['inn'] = kedr_df['inn'].map(normalize_inn)
        kedr_df['report_month'] = kedr_df['yearmm'].map(
            lambda y: f'{int(y)//100:04d}-{int(y)%100:02d}'
        )
        kedr_df = (
            kedr_df.dropna(subset=['inn', 'report_month'])
            .groupby(['inn', 'report_month'], as_index=False)['kedr_obshiy_chod']
            .max()
        )
        final_df = final_df.drop(columns=['kedr_obshiy_chod'], errors='ignore')
        final_df = final_df.merge(kedr_df, on=['inn', 'report_month'], how='left')
        print(f'Joined Kedr from {KEDR_TABLE}: rows={len(kedr_df):,}')
        has_kedr_col = True
    except Exception as exc:
        raise RuntimeError(
            'Нужен Общий ЧОД май–июль, но нет колонки в final_df и не удалось читать lake. '
            'Прогони 01_07_build_kedr_obshiy_chod_drp.ipynb (без 202608) и enrich.'
        ) from exc
else:
    print('Kedr columns already usable in final_df — skip lake join')


## 2) Списки ИНН с коммуникацией

In [ ]:
def read_inn_list(path: Path) -> set:
    if not path.exists():
        raise FileNotFoundError(path)
    inns = set()
    for line in path.read_text(encoding='utf-8').splitlines():
        s = line.strip()
        if not s or s.startswith('#'):
            continue
        s = s.split(';')[0].split(',')[0].strip()
        inn = normalize_inn(s)
        if inn:
            inns.add(inn)
    return inns


comm_by_month = {m: read_inn_list(p) for m, p in INN_LIST_PATHS.items()}
for m, inns in comm_by_month.items():
    print(f'{m}: {len(inns):,} INN')

universe = sorted(set().union(*comm_by_month.values()))
print('universe (union):', f'{len(universe):,}')
if not universe:
    raise RuntimeError('Пустой universe — проверь inn_list_*.txt')

comm_df = pd.DataFrame({'inn': universe})
comm_df['comm_2026_06'] = comm_df['inn'].isin(comm_by_month['2026-06']).astype(int)
comm_df['comm_2026_07'] = comm_df['inn'].isin(comm_by_month['2026-07']).astype(int)
comm_df['comm_2026_08'] = comm_df['inn'].isin(comm_by_month['2026-08']).astype(int)
display(comm_df.head(10))


## 3) Агрегаты ЧОД по ИНН×месяц + атрибуты клиента

In [ ]:
scope = final_df.loc[final_df['inn'].isin(universe)].copy()
print('final_df rows in universe:', f'{len(scope):,}')

chod_te = (
    scope.groupby(['inn', 'report_month'], as_index=False)['chod']
    .sum(min_count=1)
    .rename(columns={'chod': 'chod_te'})
)

chod_obsh = (
    scope.groupby(['inn', 'report_month'], as_index=False)['kedr_obshiy_chod']
    .max()
)

attr_months = ['2026-05', '2026-06', '2026-07', '2026-08']
attrs = scope.loc[scope['report_month'].isin(attr_months)].copy()
attrs['month_rank'] = attrs['report_month'].map({m: i for i, m in enumerate(attr_months)})
attrs = attrs.sort_values(['inn', 'month_rank'], ascending=[True, False])


def first_nonnull(series):
    for v in series:
        if pd.notna(v) and str(v).strip() not in {'', 'nan', 'None'}:
            return v
    return np.nan


client_attrs = (
    attrs.groupby('inn', as_index=False)
    .agg({
        'filial_rf': first_nonnull,
        'company_name': first_nonnull,
        'tariff_short': first_nonnull,
    })
)

te_wide = chod_te.pivot(index='inn', columns='report_month', values='chod_te').reset_index()
ob_wide = chod_obsh.pivot(index='inn', columns='report_month', values='kedr_obshiy_chod').reset_index()


def rename_month_cols(df, prefix):
    out = df.copy()
    out.columns = [c if c == 'inn' else f'{prefix}_{c}' for c in out.columns]
    return out


te_wide = rename_month_cols(te_wide, 'chod_te')
ob_wide = rename_month_cols(ob_wide, 'obshiy')

print('te_wide cols:', list(te_wide.columns))
print('ob_wide cols:', list(ob_wide.columns))
display(client_attrs.head(5))


## 4) Сборка wide-отчёта и Excel

In [ ]:
report = (
    comm_df
    .merge(client_attrs, on='inn', how='left')
    .merge(te_wide, on='inn', how='left')
    .merge(ob_wide, on='inn', how='left')
)

for m in NEED_CHOD_TE_MONTHS:
    col = f'chod_te_{m}'
    if col not in report.columns:
        report[col] = np.nan
for m in NEED_OBSHIY_MONTHS:
    col = f'obshiy_{m}'
    if col not in report.columns:
        report[col] = np.nan

out_df = pd.DataFrame({
    'РФ': report['filial_rf'],
    'ИНН': report['inn'],
    'Наименование клиента': report['company_name'],
    'Тариф (коротко)': report['tariff_short'],
    'ЧОД ТЭ май': report['chod_te_2026-05'],
    'Общий ЧОД май': report['obshiy_2026-05'],
    'Была коммуникация в июне': report['comm_2026_06'],
    'ЧОД ТЭ июнь': report['chod_te_2026-06'],
    'Общий ЧОД июнь': report['obshiy_2026-06'],
    'Была коммуникация в июле': report['comm_2026_07'],
    'ЧОД ТЭ июль': report['chod_te_2026-07'],
    'Общий ЧОД июль': report['obshiy_2026-07'],
    'Была коммуникация в августе': report['comm_2026_08'],
    'ЧОД ТЭ августа': report['chod_te_2026-08'],
})

flag_cols = [
    'Была коммуникация в июне',
    'Была коммуникация в июле',
    'Была коммуникация в августе',
]
assert (out_df[flag_cols].sum(axis=1) > 0).all()

out_df = out_df.sort_values(['РФ', 'ИНН'], na_position='last').reset_index(drop=True)

print('rows:', f'{len(out_df):,}')
print('with company_name:', out_df['Наименование клиента'].notna().sum())
print('August ЧОД ТЭ non-null:', out_df['ЧОД ТЭ августа'].notna().sum())
display(out_df.head(20))

xlsx_path = OUT_DIR / 'obzvon_clients_may_aug_2026.xlsx'
csv_path = OUT_DIR / 'obzvon_clients_may_aug_2026.csv'
out_df.to_excel(xlsx_path, index=False)
out_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('Saved:', xlsx_path)
print('Saved:', csv_path)


## 5) Краткая сверка покрытия

- Сколько ИНН из списков нет в `final_df` вообще
- Сколько без ЧОД ТЭ / Общего ЧОД по месяцам


In [ ]:
inns_in_final = set(final_df['inn'].dropna().unique())
missing_in_final = [i for i in universe if i not in inns_in_final]
print('INN in communication lists but absent from final_df:', f'{len(missing_in_final):,}')
if missing_in_final[:10]:
    print('examples:', missing_in_final[:10])

coverage = []
name_map = {
    '2026-05': 'ЧОД ТЭ май',
    '2026-06': 'ЧОД ТЭ июнь',
    '2026-07': 'ЧОД ТЭ июль',
    '2026-08': 'ЧОД ТЭ августа',
}
for m, col in name_map.items():
    nn = out_df[col].notna().sum()
    coverage.append({'month': m, 'metric': col, 'non_null': nn, 'null': len(out_df) - nn})

ob_map = {
    '2026-05': 'Общий ЧОД май',
    '2026-06': 'Общий ЧОД июнь',
    '2026-07': 'Общий ЧОД июль',
}
for m, col in ob_map.items():
    nn = out_df[col].notna().sum()
    coverage.append({'month': m, 'metric': col, 'non_null': nn, 'null': len(out_df) - nn})

coverage_df = pd.DataFrame(coverage)
display(coverage_df)
